In [1]:
import pandas as pd
import numpy as np
import seaborn as sb
import sklearn as sc
import nltk
import scipy
import matplotlib

In [2]:
df = pd.read_csv("../data/reviews.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.columns

Index(['review', 'sentiment'], dtype='object')

In [5]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [6]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [8]:
df['review'].iloc[0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [9]:
df['clean'] = df['review'].str.replace(r'<.*?>', ' ', regex=True).str.lower()

In [10]:
df['clean'].iloc[0]

"one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.  the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.  it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.  i would say the main appeal of the show is due to the fact that it goes where other sh

In [11]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [12]:
from nltk import word_tokenize

In [13]:
df['tokens'] = df['clean'].apply(word_tokenize)
df['tokens'].iloc[0][:20]

['one',
 'of',
 'the',
 'other',
 'reviewers',
 'has',
 'mentioned',
 'that',
 'after',
 'watching',
 'just',
 '1',
 'oz',
 'episode',
 'you',
 "'ll",
 'be',
 'hooked',
 '.',
 'they']

In [14]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english')) - {'not', 'no', 'nor'}

In [16]:
df['tokens'] = df['tokens'].apply(
    lambda tokens: [w for w in tokens if w.isalpha() and w not in stop_words]
)

df['tokens'].iloc[0][:20]

['one',
 'reviewers',
 'mentioned',
 'watching',
 'oz',
 'episode',
 'hooked',
 'right',
 'exactly',
 'happened',
 'first',
 'thing',
 'struck',
 'oz',
 'brutality',
 'unflinching',
 'scenes',
 'violence',
 'set',
 'right']

In [17]:
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [18]:
from nltk.stem import WordNetLemmatizer

In [19]:
lem = WordNetLemmatizer()

In [20]:
df['tokens'] = df['tokens'].apply(
    lambda tokens: [lem.lemmatize(x) for x in tokens]
)

df['tokens'].iloc[0][:20]

['one',
 'reviewer',
 'mentioned',
 'watching',
 'oz',
 'episode',
 'hooked',
 'right',
 'exactly',
 'happened',
 'first',
 'thing',
 'struck',
 'oz',
 'brutality',
 'unflinching',
 'scene',
 'violence',
 'set',
 'right']

In [21]:
#making strings

df['text'] = df['tokens'].apply(lambda tokens: ' '.join(tokens))

In [22]:
X = df['text']
y = df['sentiment']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [24]:
tfid = TfidfVectorizer(max_features = 5000)

X_train = tfid.fit_transform(X_train)
X_test = tfid.transform(X_test)

In [25]:
X_train.shape

(40000, 5000)

In [26]:
X_test.shape

(10000, 5000)

In [27]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [28]:
y_pred_nb = nb.predict(X_test)

In [29]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

print(accuracy_score(y_test, y_pred_nb))
print(precision_score(y_test, y_pred_nb, pos_label='positive'))
print(recall_score(y_test, y_pred_nb, pos_label='positive'))
print(classification_report(y_test, y_pred_nb))

0.8553
0.8488120950323974
0.8646
              precision    recall  f1-score   support

    negative       0.86      0.85      0.85      5000
    positive       0.85      0.86      0.86      5000

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000



In [30]:
from sklearn.svm import LinearSVC

svm = LinearSVC(random_state = 42, max_iter = 2000)
svm.fit(X_train, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,42


In [31]:
y_pred_svm = svm.predict(X_test)

print(accuracy_score(y_test, y_pred_svm))
print(precision_score(y_test, y_pred_svm, pos_label='positive'))
print(recall_score(y_test, y_pred_svm, pos_label='positive'))
print(classification_report(y_test, y_pred_svm))

0.883
0.8768201495474223
0.8912
              precision    recall  f1-score   support

    negative       0.89      0.87      0.88      5000
    positive       0.88      0.89      0.88      5000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



In [36]:
def predict_review(text):
    clean = pd.Series([text]).str.replace(r'<.*?>', ' ', regex=True).str.lower().iloc[0]
    tokens = word_tokenize(clean)
    tokens = [w for w in tokens if w.isalpha() and w not in stop_words]
    tokens = [lem.lemmatize(w) for w in tokens]
    vector = tfid.transform([' '.join(tokens)])
    return svm.predict(vector)[0]

In [37]:
print(predict_review("This movie was absolutely amazing, I loved every minute."))
print(predict_review("This was a terrible waste of time and the acting was awful."))

positive
negative
